# Radia-NGSolve Jupyter Visualization Demo

This notebook demonstrates visualization workflows for Radia-NGSolve framework in Jupyter.

**Viewers covered:**
- PyVista (inline plotting)
- NGSolve webgui (interactive)

**Requirements:**
- `pip install radia pyvista ipywidgets`
- NGSolve with webgui
- radia_ngsolve.pyd (optional for webgui demo)

In [ ]:
# Setup
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '../../src/radia'))

import radia as rad
import pyvista as pv
import numpy as np

# PyVista Jupyter backend
pv.set_jupyter_backend('static')  # or 'trame' for interactive

rad.FldUnits('m')
print("Setup complete")

## 1. Create Magnet and Export Field

In [ ]:
# Create rectangular permanent magnet
magnet = rad.ObjRecMag([0, 0, 0], [0.04, 0.04, 0.02], [0, 0, 954930])
print("Magnet created: 40x40x20 mm, Br=1.2T (NdFeB)")

# Export field to VTS
vts_file = 'field_jupyter.vts'
rad.FldVTS(magnet, vts_file,
           [-0.1, 0.1], [-0.1, 0.1], [0.02, 0.15],
           41, 41, 27, 1, 0, 1.0)
print(f"Field exported to {vts_file}")

## 2. PyVista: Scalar Field (B magnitude)

In [ ]:
# Load VTS
grid = pv.read(vts_file)

# Plot B magnitude
grid.plot(scalars='B_magnitude',
          cmap='coolwarm',
          show_edges=False,
          cpos='iso',
          window_size=[800, 600],
          jupyter_backend='static')

## 3. PyVista: Vector Field (Arrows)

In [ ]:
# Create plotter
plotter = pv.Plotter(window_size=[800, 600])

# Add mesh with transparency
plotter.add_mesh(grid, scalars='B_magnitude',
                 cmap='viridis',
                 opacity=0.3)

# Add vector arrows
arrows = grid.glyph(orient='B_field', scale='B_magnitude', factor=0.02)
plotter.add_mesh(arrows, color='black')

plotter.add_scalar_bar('B magnitude [T]')
plotter.camera_position = 'iso'
plotter.show(jupyter_backend='static')

## 4. PyVista: Slice Visualization

In [ ]:
# Create slice at z=0.05m
slice_z = grid.slice(normal='z', origin=[0, 0, 0.05])

# Plot slice with arrows
plotter = pv.Plotter(window_size=[800, 600])
plotter.add_mesh(slice_z, scalars='B_magnitude',
                 cmap='rainbow',
                 show_edges=True)

arrows = slice_z.glyph(orient='B_field', scale='B_magnitude', factor=0.02)
plotter.add_mesh(arrows, color='white')

plotter.add_scalar_bar('B magnitude [T]')
plotter.camera_position = 'xy'
plotter.show(jupyter_backend='static')

## 5. Interactive Widget: Parameter Sweep

Use ipywidgets to create interactive parameter controls.

In [ ]:
from ipywidgets import interact, FloatSlider

def visualize_field_at_height(z_slice):
    """Interactive visualization with z-slice control."""
    slice_z = grid.slice(normal='z', origin=[0, 0, z_slice])
    
    plotter = pv.Plotter(window_size=[800, 600])
    plotter.add_mesh(slice_z, scalars='B_magnitude',
                     cmap='rainbow',
                     show_edges=True)
    
    arrows = slice_z.glyph(orient='B_field', scale='B_magnitude', factor=0.02)
    plotter.add_mesh(arrows, color='white')
    
    plotter.add_scalar_bar('B magnitude [T]')
    plotter.camera_position = 'xy'
    plotter.show(jupyter_backend='static')

# Interactive slider
interact(visualize_field_at_height,
         z_slice=FloatSlider(min=0.02, max=0.15, step=0.01, value=0.05,
                            description='Z slice [m]'))

## 6. NGSolve webgui (Optional)

Requires `radia_ngsolve.pyd` to be built.

In [ ]:
# Check if radia_ngsolve is available
try:
    from radia_ngsolve import RadiaField
    from ngsolve import *
    from ngsolve.webgui import Draw
    from netgen.occ import Box, Pnt, OCCGeometry
    
    # Create mesh
    box = Box(Pnt(-0.1, -0.1, 0.02), Pnt(0.1, 0.1, 0.15))
    geo = OCCGeometry(box)
    mesh = Mesh(geo.GenerateMesh(maxh=0.02))
    
    # Radia CoefficientFunction
    B_cf = RadiaField(magnet, 'b', units='m')
    
    # Project to GridFunction
    fes = HDiv(mesh, order=2)
    B_gf = GridFunction(fes)
    B_gf.Set(B_cf)
    
    # Interactive webgui
    Draw(B_gf, mesh, 'B_field', vectors={'grid_size': 10})
    
except ImportError:
    print("radia_ngsolve not available")
    print("Build radia_ngsolve.pyd to enable NGSolve webgui demo")

## 7. Cleanup

In [ ]:
# Remove temporary VTS file
os.remove(vts_file)
print("Cleanup complete")

---

## Summary

**PyVista advantages in Jupyter:**
- Inline plotting (no external windows)
- Interactive widgets (ipywidgets)
- Scriptable and reproducible
- Fast iteration

**NGSolve webgui advantages:**
- Native NGSolve integration
- Mesh + field simultaneous display
- WebGL rendering (browser-based)

**Recommendation:** Use PyVista as default for Jupyter workflows.